# C04 · Struct layout, byte order and strict aliasing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/01_c_bare_metal/C04_struct_layout_byte_order_and_strict_aliasing.ipynb)

**Track:** C and bare-metal · **Skill axis (sheet):** C and bare-metal · **Time:** about 55 min

**Skill Hub concepts:** Struct padding and alignment; Endianness and serialization; Strict aliasing

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("C04")

## Map: how a struct sits in memory

Each member starts at a multiple of its alignment; the struct's size is rounded up to its largest alignment.

```
struct { char a; double b; char c; }        offset: 0  1..7    8..15   16  17..23
                                                    a  pad(7)  b       c   pad(7)   -> sizeof 24
struct { double b; char a; char c; }        offset: 0..7  8  9  10..15
                                                    b     a  c  pad(6)               -> sizeof 16
```

| Type | Alignment on x86-64 Linux | Alignment on Cortex-M (AAPCS) |
|---|---|---|
| `char`, `uint8_t` | 1 | 1 |
| `uint16_t` | 2 | 2 |
| `uint32_t`, `float` | 4 | 4 |
| `double`, `uint64_t` | 8 | 8 |
| `long`, pointers | 8 | 4 |

Byte order of `0x12345678` at increasing addresses:

| Order | +0 | +1 | +2 | +3 | Used by |
|---|---|---|---|---|---|
| Little-endian | 78 | 56 | 34 | 12 | Cortex-M, x86 |
| Big-endian (network order) | 12 | 34 | 56 | 78 | many wire protocols, CANopen is little-endian though |

Strict aliasing: an object may only be read through its own type, a compatible type, or a character type.
`*(uint32_t *)&f` breaks the rule; `memcpy(&u, &f, 4)` does not and compiles to the same single move.

## Task 1: predict `sizeof` and `offsetof` (x86-64), the compiler grades

In [ ]:
layout = {
    "sizeof(struct A)": None,
    "sizeof(struct B)": None,
    "offsetof(struct C, v)": None,
    "sizeof(struct C)": None,
    "sizeof(struct D)": None,
}
# struct A { char a; double b; char c; };
# struct B { double b; char a; char c; };
# struct C { uint8_t id; uint32_t v; uint16_t crc; };
# struct D { uint16_t x; uint8_t y; uint8_t z; uint16_t w; };
lab.check("layout", layout)

## Task 2: serialize without padding or host byte order
Wire format of one telemetry frame (11 bytes, big-endian):
`id` (1) | `current_mA` uint16 (2) | `position` int32 (4) | `temp_c` float IEEE-754 bits (4).

In [ ]:
%%writefile wire.c
#include <stdint.h>
#include <string.h>
#include <stddef.h>

typedef struct { uint8_t id; uint16_t current_mA; int32_t position; float temp_c; } Telemetry;

void put_u16_be(uint8_t *p, uint16_t v) {
    (void)p; (void)v; /* TODO */
}
void put_u32_be(uint8_t *p, uint32_t v) {
    (void)p; (void)v; /* TODO */
}
uint32_t get_u32_le(const uint8_t *p) {
    (void)p; return 0; /* TODO */
}
uint32_t float_bits(float f) {
    return *(uint32_t *)&f;   /* TODO: this breaks strict aliasing; fix it */
}
size_t pack_telemetry(uint8_t *out, const Telemetry *t) {
    memcpy(out, t, sizeof *t);  /* TODO: wrong on purpose (padding, host order) */
    return sizeof *t;
}

In [ ]:
lab.check("wire")

## Checkpoint: Math
**M1 (Apply).** `0xCAFEF00D` stored little-endian: the byte at offset 1?

**M2 (Analyze).** `struct { uint8_t a; uint32_t b; uint8_t c; uint16_t d; }` on Cortex-M: `sizeof`?

**M3 (Apply).** Same members reordered largest first. New `sizeof`?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Evaluate).** A teammate sends `memcpy(frame, &telemetry, sizeof telemetry)` over CAN from an STM32 to a
Linux PC. Best criticism?
- a) memcpy is slow
- b) the bytes include padding and host byte order, so the layout depends on compiler and CPU, not on a spec
- c) CAN frames cannot hold structs

**K2 (Understand).** State the strict aliasing rule, its exceptions, and why breaking it can make `-O2` code read
stale values.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `c04.h`).
3. In `roboprep/checks/c04.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()